# \(\Psi_n\) diagnostic for Zhong & Wang (2024) Theorem 3.3

Target claim (proof step):
\[
\Psi_n(\hat\zeta,\hat h)=o_p(n^{-1/2})
\quad\Leftrightarrow\quad
S_n=\sqrt{n}\,\widehat\Psi \to_p 0.
\]

After joint DPLQR-style fitting of \((\hat\theta,\hat m)\), form residuals
\(\hat r_i=Y_i-X_i\hat\theta-\hat m(Z_i)\) and the **oracle** residualized regressor
\(\tilde X_i=X_i-\varphi^*(Z_i)\). Then
\[
\widehat\Psi
=
-\frac1n\sum_i\bigl[\tau-\mathbf{1}\{\hat r_i<0\}\bigr]\tilde X_i.
\]

Also record the ordinary \(\theta\) score \(G_\theta\) (same formula with \(X_i\)) and the
**frozen-\(h\)** one-dimensional convex adjustment
\[
\hat a=\arg\min_a\frac1n\sum_i\rho_\tau(\hat r_i-a\,\tilde X_i).
\]
If the projected-score step is on track, \(\hat a\) should be near zero.

Homoskedastic PLQR toy DGP; joint Adam; fixed epochs; exact `phi_star`.

**Do not run the Monte Carlo until you intend to.** Outputs go under `./run/`.


## Configuration and imports


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import linprog

np.set_printoptions(precision=6, suppress=True)
torch.set_default_dtype(torch.float64)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
N_VALUES = [500, 1000, 2000]
Q = 30
EPOCHS = 500
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
HIDDEN = [16, 16]
BASE_SEED = 20260923
TAU = 0.5
THETA0 = 1.0
Z_DIM = 2

ROOT = Path(".").resolve()
RUN_DIR = ROOT / "run"
FIG_DIR = RUN_DIR / "figures"
RESULTS_CSV = RUN_DIR / "replication_results.csv"
SUMMARY_CSV = RUN_DIR / "summary.csv"
CONFIG_JSON = RUN_DIR / "config.json"

DEVICE = torch.device("cpu")


def replication_seed(n: int, rep: int) -> int:
    return int(BASE_SEED + 1_000_003 * n + 1_009 * rep)


print("ROOT =", ROOT)
print("N_VALUES =", N_VALUES, "Q =", Q, "EPOCHS =", EPOCHS, "HIDDEN =", HIDDEN)


## DGP helpers


In [ ]:
def phi_star(Z: np.ndarray) -> np.ndarray:
    # Exact E[X|Z] under the DGP.
    Z = np.asarray(Z, dtype=np.float64)
    z1, z2 = Z[:, 0], Z[:, 1]
    return 0.5 + 0.3 * np.sin(2.0 * np.pi * z1) + 0.3 * z1 * z2


def m0_fn(Z: np.ndarray) -> np.ndarray:
    Z = np.asarray(Z, dtype=np.float64)
    z1, z2 = Z[:, 0], Z[:, 1]
    return (
        0.5 * np.sin(2.0 * np.pi * z1)
        + 0.5 * (z2 - 0.5) ** 2
        + 0.3 * np.cos(2.0 * np.pi * z1 * z2)
    )


def generate_sample(n: int, rng: np.random.Generator) -> dict:
    Z = rng.uniform(0.0, 1.0, size=(n, 2))
    phi = phi_star(Z)
    V = rng.normal(0.0, 0.5, size=n)
    X = phi + V
    m0 = m0_fn(Z)
    eps = rng.normal(0.0, 1.0, size=n)
    Y = THETA0 * X + m0 + eps
    X_tilde = X - phi
    return {
        "Y": Y.astype(np.float64),
        "X": X.astype(np.float64),
        "Z": Z.astype(np.float64),
        "X_tilde": X_tilde.astype(np.float64),
        "phi": phi.astype(np.float64),
        "m0": m0.astype(np.float64),
    }


def check_loss(resid: np.ndarray, tau: float = TAU) -> float:
    resid = np.asarray(resid, dtype=np.float64).ravel()
    return float(np.mean(resid * (tau - (resid < 0.0).astype(np.float64))))


# Sanity: E[X|Z] = phi_star (Monte Carlo smoke, not the experiment)
_rng = np.random.default_rng(0)
_Z = _rng.uniform(0, 1, size=(200_000, 2))
_V = _rng.normal(0, 0.5, size=200_000)
_X = phi_star(_Z) + _V
print("MC check mean(X - phi_star) ≈", float(np.mean(_X - phi_star(_Z))))


## Joint Adam model (scalar \(\theta\) + MLP \(m\))

Exactly `EPOCHS` epochs. No early stopping. No validation split.


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim: int = Z_DIM, hidden: list[int] | None = None):
        super().__init__()
        if hidden is None:
            hidden = list(HIDDEN)
        layers: list[nn.Module] = []
        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            d = h
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).squeeze(-1)


class JointModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
        self.m_net = MLP()

    def forward(self, X: torch.Tensor, Z: torch.Tensor) -> torch.Tensor:
        return X * self.theta + self.m_net(Z)


def train_joint(Y: np.ndarray, X: np.ndarray, Z: np.ndarray, seed: int):
    # Joint Adam for exactly EPOCHS epochs. Returns (theta_hat, m_hat, final_loss).
    torch.manual_seed(int(seed))
    np_rng = np.random.default_rng(int(seed))

    model = JointModel().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    Y_t = torch.as_tensor(Y, dtype=torch.float64, device=DEVICE)
    X_t = torch.as_tensor(X, dtype=torch.float64, device=DEVICE)
    Z_t = torch.as_tensor(Z, dtype=torch.float64, device=DEVICE)
    n = int(Y.shape[0])

    model.train()
    final_loss = float("nan")
    for _epoch in range(1, EPOCHS + 1):
        perm = np_rng.permutation(n)
        running = 0.0
        seen = 0
        for start in range(0, n, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            idx_t = torch.as_tensor(idx, dtype=torch.long, device=DEVICE)
            pred = model(X_t[idx_t], Z_t[idx_t])
            u = Y_t[idx_t] - pred
            loss = 0.5 * torch.mean(torch.abs(u))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            running += float(loss.detach().cpu()) * len(idx)
            seen += len(idx)
        final_loss = running / max(seen, 1)

    model.eval()
    with torch.no_grad():
        theta_hat = float(model.theta.detach().cpu())
        m_hat = model.m_net(Z_t).detach().cpu().numpy().astype(np.float64)
    return theta_hat, m_hat, float(final_loss)


print("train_joint defined")


## Diagnostics: \(\widehat\Psi\), \(S_n\), \(G_\theta\), \(\hat a\)


In [ ]:
def a_hat_linprog(rhat: np.ndarray, X_tilde: np.ndarray, tau: float = TAU) -> float:
    # 1-D check-loss QR: min_a (1/n) sum rho_tau(rhat - a * X_tilde).
    r = np.asarray(rhat, dtype=np.float64).ravel()
    xt = np.asarray(X_tilde, dtype=np.float64).ravel()
    n = r.shape[0]
    # Variables: [a, u+, u-]; minimize tau*u+ + (1-tau)*u-
    # r - a*xt = u+ - u-
    c = np.concatenate([[0.0], tau * np.ones(n), (1.0 - tau) * np.ones(n)])
    A_eq = np.hstack([(-xt).reshape(n, 1), np.eye(n), -np.eye(n)])
    b_eq = r
    bounds = [(None, None)] + [(0.0, None)] * (2 * n)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(f"linprog failed: {res.message}")
    return float(res.x[0])


def replication_row(n: int, rep: int) -> dict:
    seed = replication_seed(n, rep)
    rng = np.random.default_rng(seed)
    data = generate_sample(n, rng)
    Y, X, Z, Xt = data["Y"], data["X"], data["Z"], data["X_tilde"]

    theta_hat, m_hat, final_loss = train_joint(Y, X, Z, seed)
    rhat = Y - X * theta_hat - m_hat
    ind = (rhat < 0.0).astype(np.float64)
    score = TAU - ind  # tau - 1{r<0}

    Psi_hat = -float(np.mean(score * Xt))
    G_theta = -float(np.mean(score * X))
    sqrt_n = math.sqrt(n)
    Sn = sqrt_n * Psi_hat
    sqrt_n_G = sqrt_n * G_theta
    a_hat = a_hat_linprog(rhat, Xt, TAU)

    return {
        "n": int(n),
        "rep": int(rep),
        "seed": int(seed),
        "theta_hat": float(theta_hat),
        "theta_error": float(theta_hat - THETA0),
        "Psi_hat": float(Psi_hat),
        "Sn": float(Sn),
        "abs_Sn": float(abs(Sn)),
        "G_theta": float(G_theta),
        "sqrt_n_G_theta": float(sqrt_n_G),
        "abs_sqrt_n_G_theta": float(abs(sqrt_n_G)),
        "a_hat": float(a_hat),
        "abs_a_hat": float(abs(a_hat)),
        "final_training_loss": float(final_loss),
    }


print("diagnostics defined")


## Write config (does not run Monte Carlo)


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "purpose": "Psi_n = o_p(n^{-1/2}) diagnostic for ZW Theorem 3.3",
    "N_VALUES": N_VALUES,
    "Q": Q,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "BATCH_SIZE": BATCH_SIZE,
    "HIDDEN": HIDDEN,
    "BASE_SEED": BASE_SEED,
    "TAU": TAU,
    "THETA0": THETA0,
    "DGP": {
        "Z": "Uniform(0,1)^2",
        "phi_star": "0.5 + 0.3*sin(2*pi*Z1) + 0.3*Z1*Z2",
        "X": "phi_star(Z) + N(0,0.5^2)",
        "m0": "0.5*sin(2*pi*Z1) + 0.5*(Z2-0.5)^2 + 0.3*cos(2*pi*Z1*Z2)",
        "epsilon": "N(0,1)",
        "Y": "theta0*X + m0(Z) + epsilon",
    },
    "fitting": "joint Adam (theta, m_NN); no early stopping",
    "seed_formula": "BASE_SEED + 1000003*n + 1009*rep",
}
CONFIG_JSON.write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Wrote", CONFIG_JSON)


## Monte Carlo loop (restartable)

Skips completed `(n, rep)` pairs already in `run/replication_results.csv`.


In [ ]:
def load_completed() -> set[tuple[int, int]]:
    if not RESULTS_CSV.exists():
        return set()
    df = pd.read_csv(RESULTS_CSV)
    return {(int(r.n), int(r.rep)) for r in df.itertuples(index=False)}


def append_row(row: dict) -> None:
    df = pd.DataFrame([row])
    header = not RESULTS_CSV.exists()
    df.to_csv(RESULTS_CSV, mode="a", header=header, index=False)


completed = load_completed()
print(f"Already completed: {len(completed)} rows")

for n in N_VALUES:
    for rep in range(1, Q + 1):
        key = (n, rep)
        if key in completed:
            continue
        row = replication_row(n, rep)
        append_row(row)
        completed.add(key)
        print(
            f"n={n} rep={rep}/{Q}  "
            f"theta={row['theta_hat']:.4f}  "
            f"|Sn|={row['abs_Sn']:.4f}  "
            f"|sqrt_n G|={row['abs_sqrt_n_G_theta']:.4f}  "
            f"|a|={row['abs_a_hat']:.4f}"
        )

print("Done. Wrote", RESULTS_CSV)


## Summary table and three plots


In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError("Run the Monte Carlo cell first to create replication_results.csv")

df = pd.read_csv(RESULTS_CSV)

rows = []
for n, g in df.groupby("n"):
    rows.append(
        {
            "n": int(n),
            "Q": int(len(g)),
            "mean_theta_hat": float(g["theta_hat"].mean()),
            "bias_theta": float(g["theta_error"].mean()),
            "mean_abs_Sn": float(g["abs_Sn"].mean()),
            "sd_abs_Sn": float(g["abs_Sn"].std(ddof=1)),
            "mean_abs_sqrt_n_G_theta": float(g["abs_sqrt_n_G_theta"].mean()),
            "sd_abs_sqrt_n_G_theta": float(g["abs_sqrt_n_G_theta"].std(ddof=1)),
            "mean_abs_a_hat": float(g["abs_a_hat"].mean()),
            "sd_abs_a_hat": float(g["abs_a_hat"].std(ddof=1)),
            "mean_Sn": float(g["Sn"].mean()),
            "mean_G_theta": float(g["G_theta"].mean()),
        }
    )
summary = pd.DataFrame(rows).sort_values("n")
summary.to_csv(SUMMARY_CSV, index=False)
print(summary.to_string(index=False))
print("Wrote", SUMMARY_CSV)


def _scatter_mean(ax, n_col, y_col, ylabel, title, fname):
    ax.scatter(df[n_col], df[y_col], alpha=0.35, s=18, label="reps")
    means = df.groupby(n_col)[y_col].mean()
    ax.plot(means.index, means.values, "o-", color="C1", label="mean")
    ax.set_xlabel("n")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig = ax.figure
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=120)
    plt.close(fig)


fig, ax = plt.subplots(figsize=(5.5, 3.8))
_scatter_mean(ax, "n", "abs_Sn", r"$|\sqrt{n}\widehat\Psi|$", r"$|\sqrt{n}\Psi_{\mathrm{hat}}|$ vs $n$", "abs_Sn_vs_n.png")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
_scatter_mean(
    ax,
    "n",
    "abs_sqrt_n_G_theta",
    r"$|\sqrt{n} G_\theta|$",
    r"$|\sqrt{n} G_\theta|$ vs $n$",
    "abs_sqrt_n_Gtheta_vs_n.png",
)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
_scatter_mean(ax, "n", "abs_a_hat", r"$|\hat a|$", r"$|\hat a|$ vs $n$", "abs_ahat_vs_n.png")

print("Wrote figures under", FIG_DIR)


## Short summary answers


In [ ]:
s = summary.sort_values("n")
ns = s["n"].to_numpy()
m_sn = s["mean_abs_Sn"].to_numpy()
m_g = s["mean_abs_sqrt_n_G_theta"].to_numpy()
m_a = s["mean_abs_a_hat"].to_numpy()


def _decreases(vals) -> str:
    if len(vals) < 2:
        return "insufficient n grid"
    if vals[-1] < 0.8 * vals[0]:
        return "YES — mean absolute value shrinks from smallest to largest n"
    if vals[-1] < vals[0]:
        return "WEAK YES — shrinks mildly"
    return "NO — does not clearly decrease toward zero"


print("=== Psi_n / Theorem 3.3 diagnostic ===")
print("1. Does |sqrt(n) Psi_hat| decrease toward zero?")
print("  ", _decreases(m_sn))
print("   mean |Sn| by n:", {int(n): float(v) for n, v in zip(ns, m_sn)})
print("2. Is the ordinary |sqrt(n) G_theta| already small?")
g_last = float(m_g[-1])
if g_last < 0.25:
    print(f"   YES-ish — at n={int(ns[-1])}, mean|sqrt(n)G|={g_last:.4f}")
elif g_last < 1.0:
    print(f"   MODERATE — at n={int(ns[-1])}, mean|sqrt(n)G|={g_last:.4f}")
else:
    print(f"   NO — at n={int(ns[-1])}, mean|sqrt(n)G|={g_last:.4f}")
print("   mean |sqrt(n)G| by n:", {int(n): float(v) for n, v in zip(ns, m_g)})
print("3. Does frozen-h |a_hat| vanish?")
print("  ", _decreases(m_a))
print("   mean |a_hat| by n:", {int(n): float(v) for n, v in zip(ns, m_a)})
